# Fine-tune LLM using LoRA and QLoRA

In [ ]:
%cd /content
!rm -rf LoRA-QLoRA-studies
!git clone https://github.com/shosakaue0808/LoRA-QLoRA-studies.git
%cd /content/LoRA-QLoRA-studies

In [ ]:
!pip install transformers peft bitsandbytes python-dotenv datasets

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import prepare_model_for_kbit_training
from huggingface_hub import login
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from dotenv import load_dotenv
import os
# my files
import importlib

from src.data_prep import GSMDataset
from src.train_utils import load_checkpoint, train
from src.lora_layer import attach_Lora_to_Linear

## login Hugging face

In [ ]:
load_dotenv()
token = os.getenv("HF_TOKEN")
login(token)

In [ ]:
# Set device (use GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load Model

# LoRA base

In [ ]:
model_id = "meta-llama/Llama-3.2-1B"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
base_model_lora = AutoModelForCausalLM.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
base_model_lora.config.pad_token_id = tokenizer.pad_token_id

# QLoRA base

In [ ]:
from transformers import BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
base_model_qlora = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config)
base_model_qlora.model_name = model_id
base_tokenizer_qlora = AutoTokenizer.from_pretrained(model_id)
base_tokenizer_qlora.pad_token = base_tokenizer_qlora.eos_token
base_model_qlora.config.pad_token_id = base_tokenizer_qlora.pad_token_id

In [ ]:
for name, p in base_model_lora.named_parameters():
    if p.requires_grad:
        print(name, p.shape)

In [ ]:
#Set all weights freeze
for param in base_model_lora.parameters():
    param.requires_grad = False

# Attach LoRA layer to all linear layers in base model

In [ ]:
print(base_model_lora.model.layers)

In [ ]:
attach_Lora_to_Linear(base_model_lora, rank=16, alpha=16)

In [ ]:
print(base_model_lora)

In [ ]:
for name, p in base_model_lora.named_parameters():
    if p.requires_grad:
        print(name, p.shape)

# Fine-tune process

In [ ]:
base_model_lora = base_model_lora.to(device)
learning_rate = 1e-4
optimizer = torch.optim.AdamW(base_model_lora.parameters(), lr=learning_rate)
num_epochs = 2